## Imports

- `evaluate_filter` : compare le filtre aux étiquettes NotUsable des biologistes et écrit le rapport par vidéo ;
- `config` : chemins et seuils du filtre.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent))

from pipeline.filtrage import config
from pipeline.filtrage.evaluate import evaluate_filter

## Évaluation du filtre

Lit le CSV écrit par `filter_frames.ipynb`, écrit le rapport par vidéo, puis affiche le rappel et la précision.

Relancer `filter_frames.ipynb` d'abord si les seuils ont changé.

In [ ]:
resultats = evaluate_filter()

print()
print(f"TEXTURE_THRESHOLD = {config.TEXTURE_THRESHOLD}")
print(f"VIDEO_REJECT_PERCENT = {config.VIDEO_REJECT_PERCENT}")

## Aperçu du rapport par vidéo

Une ligne par vidéo : pourcentage de frames rejetées, répartition par cause, prédiction du filtre et étiquette des biologistes.

In [ ]:
rapport = pd.read_csv(config.PATH_TO_VIDEO_REPORT_CSV)

print(rapport.dtypes)
rapport.head(10)

## Erreurs du filtre

- **Faux négatifs** : vidéos NotUsable selon les biologistes, mais gardées par le filtre ;
- **Faux positifs** : vidéos utilisables selon les biologistes, mais rejetées par le filtre.

Les vidéos sans étiquette ne sont dans aucun des deux groupes.

In [ ]:
# label_biologistes est vide pour une vidéo non étiquetée : pandas y met NaN,
# qui n'est égal ni à True ni à False. Ces vidéos sont donc exclues des deux groupes
etiquette = rapport["label_biologistes"]
prediction = rapport["prediction_not_usable"]

faux_negatifs = rapport[etiquette.eq(True) & prediction.eq(False)]
faux_positifs = rapport[etiquette.eq(False) & prediction.eq(True)]

print(f"Faux négatifs : {len(faux_negatifs)}")
print(f"Faux positifs : {len(faux_positifs)}")

### Faux négatifs

Triés par pourcentage de frames rejetées : en haut, les vidéos proches du seuil `VIDEO_REJECT_PERCENT` ; en bas, celles où le filtre n'a presque rien détecté.

In [ ]:
faux_negatifs.sort_values("pct_rejetees", ascending=False)

### Faux positifs

Les colonnes `pct_<cause>` montrent quelle règle a rejeté ces vidéos à tort.

In [ ]:
faux_positifs.sort_values("pct_rejetees", ascending=False)